# Cortex v2 Training (Kaggle - T4 GPU)
**Setup:**
1. Go to https://kaggle.com → Create → New Notebook
2. **Settings → Accelerator → GPU T4**
3. **Add Data → Upload** → select `cortex_train_data.zip`
4. Paste this notebook, then Run All

In [ ]:
!pip install torch numpy tqdm tokenizers -q
print('OK')

In [ ]:
import zipfile, os, glob

zips = glob.glob('/kaggle/input/**/cortex_train_data.zip', recursive=True) or [f for f in os.listdir('.') if f.endswith('.zip')]
if not zips:
    print('ERROR: zip not found. Upload via Add Data button.')
else:
    zip_path = zips[0]
    print(f'Found: {zip_path}')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('.')
    print('Extracted. Files:')
    for f in sorted(os.listdir('.')):
        if f.endswith('.py') or f.startswith('data'):
            print(f'  {f}')

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0)}')
import model, train_v2
print('Modules loaded OK')

In [ ]:
!python train_v2.py --data-dir data --batch-size 8 --seq-len 256 --lr 3e-4 --steps 50000 --warmup 2000 --save-every 10000 --val-every 1000 --checkpoint-dir checkpoints --d-model 320 --n-layers 6 --n-heads 10 --d-ff 1280
print('TRAINING DONE')

In [ ]:
import os
if not os.path.exists('checkpoints'):
    print('ERROR: checkpoints not found')
else:
    for f in sorted(os.listdir('checkpoints')):
        path = os.path.join('checkpoints', f)
        print(f'{f}  ({os.path.getsize(path)/1e6:.1f} MB)')
    print('\nTo download: click the Data tab on the right → checkpoints/ → download')

In [ ]:
if os.path.exists('checkpoints/final.pt'):
    !python generate.py --checkpoint checkpoints/best.pt --data-dir data --prompt 'Once upon a time' --max-tokens 64 --temperature 0.6 --top-k 50 --repetition-penalty 1.2
else:
    print('No checkpoint')